# Using the ML–EHR notebooks with synthetic data

This tutorial helps readers use the repository’s feature-construction and Random
Forest training and evaluation code with synthetic input data. Run the cells
from top to bottom to follow the workflow. This example uses 120 entirely synthetic subjects
(40 simulated cases and 80 controls), and shows each input and output for feature
construction and Random Forest retraining and evaluation. It uses the original functions in
`feature_construction_for_prediction_window.ipynb` and `evaluation_retrain_parad.ipynb`.

In a terminal, from the repository root:
```bash
conda activate ml-ehr-toy
python -m pip install jupyterlab
python -m jupyterlab toy_example/toy_example.ipynb
```
If you have not created the environment, follow the README setup first. Select the
Python kernel from that conda environment. Outputs are saved in `outputs/`.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
import joblib

# Jupyter may start in either the example directory or the repository root.
HERE = next((p / 'toy_example' for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'toy_example' / 'notebook_utils.py').is_file()), None)
if HERE is None:
    raise RuntimeError('Open this notebook from inside the ml-ehr-main repository.')
sys.path.insert(0, str(HERE))
from notebook_utils import notebook_functions, prepare_features, feature_dictionary, retrain_functions, toy_rf_candidates
OUTPUT = HERE / 'outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Input directory: toy_example/data/')
print('Output directory: toy_example/outputs/')


Input directory: toy_example/data/
Output directory: toy_example/outputs/


## 1. Inspect the input tables

The toy cohort supplies subject IDs, index dates and simulated labels. Diagnosis,
medication, laboratory and vital-sign CSVs have the columns expected by the original
`process_features` function. Demographics include age at index, sex, race and ethnicity; birth dates allow conversion to age at the prediction window cutoff.
Diagnosis uses Phecodes, medication uses RxNorm ingredient codes, and laboratory tables include LOINC codes and simulated numeric values. All values here are synthetic.


In [2]:
def read_table(name, dates):
    return pd.read_csv(HERE / 'data' / f'{name}.csv', parse_dates=dates, dtype={'phecode': str, 'lab_loinc': str})

cohort = read_table('cohort', ['INDEX_DATE'])
demo = read_table('demographics', ['birth_date']).set_index('patid')
dx = read_table('diagnosis', ['dx_date_fill'])
rx = read_table('medication', ['rx_start_date'])
lab = read_table('laboratory', ['specimen_date'])
vital = read_table('vital', ['measure_date'])
for name, table in [('Cohort', cohort), ('Demographics', demo.reset_index()),
                    ('Diagnosis', dx), ('Medication', rx), ('Laboratory_test', lab), ('Vitals', vital)]:
    print(name, table.shape)
    display(table.head(3))


Cohort (120, 4)


,patid,INDEX_DATE,label,split
0,900001,2020-01-01,1,hold_out_test
1,900002,2020-01-01,1,train
2,900003,2020-01-01,1,train


Demographics (120, 6)


,patid,birth_date,age,sex,race,ethnicity
0,900001,1937-06-15,82.546,Female,White,Hispanic or Latino
1,900002,1946-06-15,73.547,Male,Black or African American,Not Hispanic or Latino
2,900003,1956-06-15,63.546,Female,Asian,Not Hispanic or Latino


Diagnosis (632, 4)


...

,patid,rx_start_date,rxcui_ing,medication_name
0,900001,2008-01-01,29046,lisinopril
1,900001,2014-01-01,6809,metformin
2,900001,2014-01-01,83367,atorvastatin


Laboratory_test (1080, 9)


,patid,specimen_date,Class,flag,lab_loinc,result_num,result_unit,norm_range_low,norm_range_high
0,900001,2008-01-01,hemoglobin_a1c,ablow,4548-4,3.85,%,4.0,5.6
1,900001,2008-01-01,glucose,ablow,2345-7,65.93,mg/dL,70.0,110.0
2,900001,2008-01-01,total_cholesterol,nor,2093-3,190.11,mg/dL,100.0,200.0


Vitals (360, 6)


,patid,measure_date,ht,wt,systolic,diastolic
0,900001,2008-01-01,66,174,137,88
1,900001,2014-01-01,66,158,124,78
2,900001,2018-01-01,66,135,108,94


## 2. Use the original analysis notebooks

The original analysis notebooks cover data preprocessing, cohort preprocessing, feature preprocessing, feature construction for different prediction windows, and model evaluation. This toy example provides synthetic input tables and a step-by-step tutorial to help readers run the feature-construction and Random Forest training and evaluation code from those notebooks, inspect intermediate feature matrices, and understand the expected inputs and outputs. The following cells load and call the relevant functions from `feature_construction_for_prediction_window.ipynb` and `evaluation_retrain_parad.ipynb`.


In [3]:
feature_functions = notebook_functions(
    'feature_construction_for_prediction_window.ipynb', ['get_early_date', 'process_features'])
evaluation_functions = retrain_functions()
process_features = feature_functions['process_features']
preprocess = evaluation_functions['preprocess']
calculate_metrics = evaluation_functions['calculate_metrics']
print('Loaded original feature and RF retrain functions.')


Loaded original feature and RF retrain functions.


## 3. Load the split

The `split` column in `cohort.csv` assigns each subject to `train` or
`hold_out_test` before any prediction-window feature construction. This stratified
cohort-level split contains: 84 training and 36 hold-out testing subjects.
The same assignments are used for every prediction window; changing the window
does not resplit patients. The hold-out subjects are not used to fit the age scaler
or train the model.


In [4]:
assert cohort.patid.is_unique
assert set(cohort['split']) == {'train', 'hold_out_test'}
cases = cohort.loc[cohort.label == 1, ['patid', 'INDEX_DATE']]
controls = cohort.loc[cohort.label == 0, ['patid', 'INDEX_DATE']]
train_ids = cohort.loc[cohort['split'] == 'train', 'patid']
test_ids = cohort.loc[cohort['split'] == 'hold_out_test', 'patid']
splits = cohort.copy()
display(splits.groupby(['split', 'label']).size().unstack(fill_value=0))


label,0,1
split,,
hold_out_test,24,12
train,56,28


## 4. Select the prediction window and construct the feature matrix

Call the original function with cases, controls, the selected window and the four
EHR domain tables. The example `prediction_window = 0` means one day before the index date; positive integers
mean years. Try 1, 2, 5 or 10 and rerun from the next cell onward. 

In [5]:
prediction_window = 0
assert prediction_window in [0, 1, 2, 5, 10]
X, y, eligible = process_features(
    cases, controls, [prediction_window], dx.copy(), rx.copy(), lab.copy(), vital.copy(), common=False)
X = prepare_features(X, demo)
assert len(eligible) == 120
assert '250.1_dx' not in X.columns
assert np.isfinite(X.drop(columns='patid').to_numpy()).all()
feature_matrix = X.assign(label=y)
feature_matrix.to_csv(OUTPUT / 'feature_matrix.csv', index=False)
print('Feature matrix:', feature_matrix.shape, '(ID + 26 predictors + label)')
display(feature_matrix.head(10))
display(feature_dictionary(X.drop(columns='patid').columns))


----****Constructing the feature matrix for computable phenotype 1 for a 0-year prediction window ...
	all patients:  120

There are a total of 120 samples, with 40 cases and 80 controls.
	-- define date offset <DateOffset: days=1>
	-- define date offset_early <DateOffset: years=1> 


#### Processing the diganosis data ... (632, 4)
	--Dropped EHR after offset, now rows and unique people:  (512, 4) 120
	Get first date...

#### Processing the medication data ... (495, 4)
	--Dropped EHR after offset, now  (495, 4) 120
	Get first date...


feature_construction_for_prediction_window.ipynb:39: FutureWarning: The provided callable <built-in function min> is currently using np.minimum.reduce. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string np.minimum.reduce instead.
feature_construction_for_prediction_window.ipynb:64: FutureWarning: The provided callable <built-in function min> is currently using np.minimum.reduce. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string np.minimum.reduce instead.



#### Processing the measurement data ... (1080, 9)
	--Dropped EHR after offset, now  (1080, 9) 120
	Getting recent measurements......
	-- Recent measurements (360, 9)

#### Processing the vital data ... (360, 6)
	--Dropped EHR after offset, now  (360, 6) 120
----keep last date ht (120, 3)
----keep last date wt (120, 3)
----keep last date systolic (120, 3)
----keep last date diastolic (120, 3)
--computing bmi


  0%|          | 0/120 [00:00<?, ?it/s]

100%|██████████| 120/120 [00:00<00:00, 305410.49it/s]

...

Generating the final feature matrix ...


The dimension of the feature matrix is (120, 21).

	Cases:  40 Controls:  80 Ratio:  2.0
--------------------------- Done -------------------------------------



Feature matrix: (120, 28) (ID + 26 predictors + label)


,patid,250.2_dx,401.1_dx,6809_rx,29046_rx,83367_rx,bmi_abhigh_measurement,bmi_nor_measurement,dbp_abhigh_measurement,dbp_nor_measurement,...,total_cholesterol_abhigh_measurement,total_cholesterol_ablow_measurement,total_cholesterol_nor_measurement,age_at_prediction_window,sex_Male_demographic,race_Black or African American_demographic,race_Asian_demographic,race_Other_demographic,ethnicity_Hispanic or Latino_demographic,label
0,900001,1,1,1,1,1,0,1,1,0,...,0,0,1,82.543463,0,0,0,0,1,1
1,900002,0,1,1,0,1,0,1,1,0,...,0,0,1,73.544148,1,1,0,0,0,1
2,900003,1,1,0,1,1,0,1,0,1,...,1,0,0,63.542779,0,0,1,0,0,1
3,900004,1,1,1,1,0,0,1,1,0,...,1,0,0,76.544832,1,0,0,1,0,1
4,900005,1,1,1,1,1,0,1,0,1,...,0,0,1,72.544832,0,0,0,0,0,1
5,900006,1,1,1,1,1,0,1,0,1,...,0,0,1,73.544148,1,1,0,0,1,1
6,900007,1,1,1,1,1,1,0,0,1,...,0,0,1,73.544148,0,0,1,0,0,1
7,900008,1,1,1,1,0,0,1,0,1,...,0,0,1,69.544148,1,0,0,1,0,1
8,900009,1,1,1,1,1,1,0,1,0,...,0,0,1,84.544832,0,0,0,0,0,1
9,900010,1,1,1,1,1,1,0,0,1,...,0,0,1,60.544832,1,1,0,0,0,1


,feature,type,description
0,250.2_dx,binary_indicator,Type 2 diabetes (Phecode 250.2)
1,401.1_dx,binary_indicator,Essential hypertension (Phecode 401.1)
2,6809_rx,binary_indicator,Metformin ingredient (RxNorm 6809)
3,29046_rx,binary_indicator,Lisinopril ingredient (RxNorm 29046)
4,83367_rx,binary_indicator,Atorvastatin ingredient (RxNorm 83367)
5,bmi_abhigh_measurement,binary_indicator,bmi abhigh measurement
6,bmi_nor_measurement,binary_indicator,bmi nor measurement
7,dbp_abhigh_measurement,binary_indicator,dbp abhigh measurement
8,dbp_nor_measurement,binary_indicator,dbp nor measurement
9,glucose_abhigh_measurement,binary_indicator,glucose abhigh measurement


## 5. Prepare the model inputs

Use the original evaluation notebook's `preprocess` function. It fits the age scaler
on training subjects and applies the same transformation to test subjects. Binary
indicators remain unchanged. Displayed model inputs contain only the 26 predictors.


In [6]:
train = X.patid.isin(train_ids)
test = X.patid.isin(test_ids)
features = X.drop(columns='patid')
xtrain, xtest, scaler = preprocess(features.loc[train].copy(), features.loc[test].copy())
ytrain, ytest = y[train], y[test]
xtrain.assign(patid=X.loc[train, 'patid'], label=ytrain).to_csv(
    OUTPUT / 'model_input_train.csv', index=False)
xtest.assign(patid=X.loc[test, 'patid'], label=ytest).to_csv(
    OUTPUT / 'model_input_test.csv', index=False)
print('Training:', xtrain.shape, 'Test:', xtest.shape)
display(xtrain.head())


	preprocess column ['age_at_prediction_window']
Training: (84, 26) Test: (36, 26)


,250.2_dx,401.1_dx,6809_rx,29046_rx,83367_rx,bmi_abhigh_measurement,bmi_nor_measurement,dbp_abhigh_measurement,dbp_nor_measurement,glucose_abhigh_measurement,...,sbp_nor_measurement,total_cholesterol_abhigh_measurement,total_cholesterol_ablow_measurement,total_cholesterol_nor_measurement,age_at_prediction_window,sex_Male_demographic,race_Black or African American_demographic,race_Asian_demographic,race_Other_demographic,ethnicity_Hispanic or Latino_demographic
1,0,1,1,0,1,0,1,1,0,1,...,0,0,0,1,-0.035872,1,1,0,0,0
2,1,1,0,1,1,0,1,0,1,1,...,0,1,0,0,-1.347688,0,0,1,0,0
3,1,1,1,1,0,0,1,1,0,0,...,0,1,0,0,0.357709,1,0,0,1,0
4,1,1,1,1,1,0,1,0,1,1,...,0,0,0,1,-0.166946,0,0,0,0,0
5,1,1,1,1,1,0,1,0,1,0,...,0,0,0,1,-0.035872,1,1,0,0,1


## 6. Model training and evaluation using Random Forest 

This cell uses functions from `evaluation_retrain_parad.ipynb`:

1. `build_RF_model(42)` shows the original RF search configuration.
2. `evaluate_params(params, inner_cv, xtrain, ytrain)` trains RFs inside the training
   cohort and scores each parameter setting.
3. Select the best setting, then fit a final RF on all training subjects, following
   the final-fit step in `nested_cv_pipeline_parallel`.
4. `evaluate_ensemble_model` computes hold-out evaluation results with `plot=False`.

For a quick toy run, use two parameter settings (50 or 100 trees) and three inner
folds. The full notebook uses five inner/outer folds and a larger search. This
example uses the cohort hold-out rather than running its full outer-CV loop.
Hold-out labels are used only for final evaluation, not parameter selection.
Only AUROC and average precision are displayed: the original metric function's
other four basic metrics currently contain placeholder values.


In [7]:
_, original_grid = evaluation_functions['build_RF_model'](42)
print('Original RF grid:', original_grid)
candidates = toy_rf_candidates()
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
search_results = [evaluation_functions['evaluate_params'](
    params, inner_cv, xtrain.copy(), ytrain, random_seed=42) for params in candidates]
search_table = pd.DataFrame([dict(**params, inner_cv_macro_f1=score)
                             for score, params in search_results])
display(search_table)
best_score, best_params = max(search_results, key=lambda result: result[0])
print('Selected RF parameters:', best_params)
model = RandomForestClassifier(random_state=42, n_jobs=-1, **best_params)
model.fit(xtrain, ytrain)
probabilities = model.predict_proba(xtest.to_numpy())[:, 1]
_, evaluation_results = evaluation_functions['evaluate_ensemble_model'](
    xtest, ytest, model, CP_num=1, prediction_window=prediction_window,
    N=30, feature_map={}, model_name='Random Forest', plot=False)
auc, ap = evaluation_results[:2]
predictions = pd.DataFrame({'patid': X.loc[test, 'patid'], 'label': ytest,
                            'probability': probabilities})
metrics = pd.DataFrame([{'window': prediction_window, 'n_train': len(xtrain),
                         'n_test': len(xtest), 'n_features': features.shape[1],
                         'auroc': auc, 'average_precision': ap}])
predictions.to_csv(OUTPUT / 'predictions.csv', index=False)
metrics.to_csv(OUTPUT / 'metrics.csv', index=False)
joblib.dump({'model': model, 'age_scaler': scaler, 'feature_columns': features.columns.tolist(),
             'best_params': best_params, 'prediction_window': prediction_window}, OUTPUT / 'rf_model.joblib')
display(predictions.head(10))
display(metrics)


max_depth option: [3, 7, 11, 15, 20]
Original RF grid: {'n_estimators': [1000, 3000, 5000], 'max_depth': [3, 7, 11, 15, 20], 'min_samples_split': [5, 10, 40], 'max_features': ['sqrt']}


Done evaluating params... {'n_estimators': 50, 'max_depth': 3, 'min_samples_split': 5, 'max_features': 'sqrt'}


Done evaluating params... {'n_estimators': 100, 'max_depth': 6, 'min_samples_split': 5, 'max_features': 'sqrt'}


,n_estimators,max_depth,min_samples_split,max_features,inner_cv_macro_f1
0,50,3,5,sqrt,0.410350
1,100,6,5,sqrt,0.470359


Selected RF parameters: {'n_estimators': 100, 'max_depth': 6, 'min_samples_split': 5, 'max_features': 'sqrt'}
[0.5138888888888888, 0.4278413670848343, 0.1, 0.1, 0.1, 0.1, 0.08333333333333333, 0.08333333333333333, 1.0, 1.0]


,patid,label,probability
0,900001,1,0.159075
6,900007,1,0.239269
7,900008,1,0.434128
8,900009,1,0.106084
10,900011,1,0.306649
11,900012,1,0.395552
14,900015,1,0.283173
23,900024,1,0.597796
24,900025,1,0.507660
28,900029,1,0.339289


,window,n_train,n_test,n_features,auroc,average_precision
0,0,84,36,26,0.513889,0.427841
